<a href="https://colab.research.google.com/github/RAHULRAJPAL75/Rahul_IN226051102_NLP/blob/main/Copy_of_Task_3_Build_a_Chatbot_using_Hugging_Face_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ipywidgets --upgrade
!jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [ ]:
!pip install transformers torch


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def rule_based_response(user_input):
    """
    This function handles common queries with fixed responses
    to improve chatbot accuracy.
    """
    user_input = user_input.lower()

    if "hello" in user_input:
        return "Hello! Nice to meet you. How can I assist you today?"

    elif "what is artificial intelligence" in user_input or "what is ai" in user_input:
        return "Artificial Intelligence refers to the simulation of human intelligence by machines."

    elif "who created python" in user_input:
        return "Python was created by Guido van Rossum in 1991."

    elif "thank" in user_input:
        return "You're welcome! Feel free to ask more questions."

    return None

In [ ]:
def generate_response(user_input, chat_history_ids):
    """
    Generates response using DialoGPT transformer model.
    """
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=300,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [ ]:
def chatbot():
    """
    Main chatbot loop for continuous conversation.
    """
    print("🤖 Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

    chat_history_ids = None

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("🤖 Chatbot: Goodbye! Have a great day!")
            break
        rule_response = rule_based_response(user_input)

        if rule_response:
            print("🤖 Chatbot:", rule_response)
            continue
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        print("🤖 Chatbot:", response)

In [ ]:
chatbot()

🤖 Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

You: HI
🤖 Chatbot: Hi!
You: Hello
🤖 Chatbot: Hello! Nice to meet you. How can I assist you today?
You: What is AI?
🤖 Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines.
You: Who created Python?
🤖 Chatbot: Python was created by Guido van Rossum in 1991.
You: Tell me about space
🤖 Chatbot: I'm a space man
You: exit
🤖 Chatbot: Goodbye! Have a great day!
